![Module 6: Deploy to AgentCore Runtime](../images/module-6-deploy.svg)

# Module 6: Deploy to AgentCore Runtime

> Same `agent.py` → managed container on AWS with its own ARN. Invokable from Lambda, mobile, another agent — anywhere with AWS credentials.

📖 Full explainer: see the companion page in `content/06-deploy-agentcore/`.  
💻 Standalone deploy script: `code/step-05-deploy/deploy.sh`


## Install the new AgentCore CLI

This notebook uses the official [`@aws/agentcore`](https://github.com/aws/agentcore-cli) CLI — an **npm package** (Node 20+ required).

> The CLI needs Node 20+. If your environment ships an older Node, run (in a terminal, not this notebook):
>
> ```bash
> nvm install 20 && nvm use 20
> npm install -g @aws/agentcore
> agentcore --version
> ```
>
> Then come back to this notebook.


In [ ]:
!nvm install 20 && nvm use 20
!npm install -g @aws/agentcore


In [ ]:
!agentcore --version   # expect 0.13.0+


## Configure deployment

Set your AgentCore Memory ID and (optionally) a runtime role ARN.

In [ ]:
import os
import boto3
ssm = boto3.client('ssm')
MODEL_ID = ssm.get_parameter(Name='/aim302/model-id')['Parameter']['Value']  # Claude Sonnet 4.5
REGION = ssm.get_parameter(Name='/aim302/region')['Parameter']['Value']
MEMORY_ID = ssm.get_parameter(Name='/aim302/memory-id')['Parameter']['Value']
RUNTIME_ROLE_ARN = ssm.get_parameter(Name='/aim302/runtime-role-arn')['Parameter']['Value']


os.environ["AWS_REGION"] = REGION
os.environ["BEDROCK_AGENTCORE_MEMORY_ID"] = MEMORY_ID

print(f"Memory:       {MEMORY_ID}")
print(f"Runtime role: {RUNTIME_ROLE_ARN or '(auto-created by CLI)'}")
print(f"Model:        {MODEL_ID}")
print(f"Region:       {REGION}")

## 1. Scaffold the AgentCore project

In [ ]:
!agentcore create --name Aim308Nb --no-agent --defaults --skip-install --skip-python-setup --skip-git


## 2. Register our agent as BYO

The agent code lives in `../code/step-05-deploy/agent.py`. We point AgentCore at it without copying.


In [ ]:
import subprocess
code_location = os.path.abspath('../code/step-05-deploy')
print('code_location:', code_location)

subprocess.run([
    'agentcore', 'add', 'agent',
    '--name', 'aim308_nb_agent',
    '--type', 'byo',
    '--language', 'Python',
    '--framework', 'Strands',
    '--model-provider', 'Bedrock',
    '--code-location', code_location,
    '--entrypoint', 'agent.py',
    '--protocol', 'HTTP',
], cwd='Aim308Nb', check=True)


## 3. Pass runtime env vars via `.env.local`

In [ ]:
env_file = 'Aim308Nb/agentcore/.env.local'
with open(env_file, 'a') as f:
    f.write(f'AWS_REGION={REGION}\n')
    f.write(f'BEDROCK_AGENTCORE_MEMORY_ID={MEMORY_ID}\n')

print('--- .env.local ---')
print(open(env_file).read())


## 4. Deploy

First install CDK deps once, then deploy. Expect 60-120s.


In [ ]:
!cd Aim308Nb/agentcore/cdk && npm install --no-audit --no-fund --loglevel=error


In [ ]:
!cd Aim308Nb && agentcore deploy -y


## 5. Invoke the deployed agent

In [ ]:
!cd Aim308Nb && agentcore invoke 'hello, introduce yourself and list your capabilities'


In [ ]:
# Session continuity — pass --session-id to accumulate memory across turns
!cd Aim308Nb && agentcore invoke --session-id demo-1 'I prefer python'


In [ ]:
!cd Aim308Nb && agentcore invoke --session-id demo-1 'what do you remember about me?'


## 6. Observability

In [ ]:
!cd Aim308Nb && agentcore status


In [ ]:
!cd Aim308Nb && agentcore logs --since 5m


🎉 **Congrats — you built a fully autonomous, self-improving agent and deployed it to AWS.**

## Cleanup

Remove the deployed agent and its resources when you're done:


In [ ]:
!cd Aim308Nb && agentcore remove agent --name aim308_nb_agent
!cd Aim308Nb && agentcore deploy -y   # CDK tears down the runtime
